# 07 Generate summary candidates

Generate ten outputs per paper and retain the first five distinct summaries. The additional five outputs are backups for duplicate text.

**Kernel:** `.venv` (`.venv/Scripts/python.exe`).


In [1]:
from pathlib import Path
import sys
import os
import json
import pandas as pd
from IPython.display import display

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'configs' / 'config.yaml').is_file()
             and (path / 'outputs' / 'baseline.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the csc173-midterm repository.')
os.chdir(ROOT)
if Path(sys.prefix).resolve() != (ROOT / '.venv').resolve():
    raise RuntimeError('Select the .venv notebook kernel, then restart.')
import hashlib

RUN = ROOT / 'outputs' / 'faithfulness' / 'run_01'
N = 618

print('Run folder:', RUN)
print('Kernel:', sys.executable)


Run folder: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\outputs\faithfulness\run_01
Kernel: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\.venv\Scripts\python.exe


## Function definitions


### Shared helper functions

Reading, validation, file saving, and resumption helpers.


In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def require(condition, message):
    if not condition:
        raise ValueError(message)


def nonblank(frame, columns):
    for column in columns:
        require(column in frame, f'Missing column: {column}')
        require(frame[column].map(lambda x: isinstance(x, str) and bool(x.strip())).all(),
                f'Blank or invalid text in {column}')


def reference_list(text):
    values = json.loads(text)
    require(isinstance(values, list) and len(values) > 0, 'Missing reference list')
    require(all(isinstance(x, str) and bool(x.strip()) for x in values), 'Invalid reference')
    return values


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False, lineterminator='\n')
    os.replace(temporary, path)


def write_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def manifest(path, value):
    # Reject changed inputs/configuration instead of reusing incompatible partial results.
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False)
    if path.exists():
        old = json.dumps(json.loads(path.read_text(encoding='utf-8')),
                         sort_keys=True, ensure_ascii=False, allow_nan=False)
        require(old == encoded, f'Protocol changed: {path}. Use a new run folder.')
    else:
        write_json(value, path)


def load_eval():
    frame = read_csv(RUN / 'evaluation_pairs.csv')
    nonblank(frame, ['sample_id', 'abstract', 'baseline_summary', 'human_tldrs'])
    require(len(frame) == N and frame.sample_id.is_unique, 'Expected 618 unique papers')
    frame.human_tldrs.map(reference_list)
    return frame


def validate_candidates(frame, evaluation):
    nonblank(frame, ['sample_id', 'candidate_id', 'abstract', 'candidate_summary'])
    require(len(frame) == N * 5, 'Expected 3090 candidate rows')
    require(set(frame.sample_id) == set(evaluation.sample_id), 'Candidate ID set differs')
    require(not frame.duplicated(['sample_id', 'candidate_id']).any(), 'Duplicate keys')
    for sample_id, group in frame.groupby('sample_id', sort=False):
        require(set(group.candidate_id.astype(str)) == set(map(str, range(5))),
                f'Candidate IDs must be 0..4: {sample_id}')
        normalized = group.candidate_summary.map(lambda text: ' '.join(text.split()))
        require(normalized.nunique() == 5, f'Non-distinct candidates: {sample_id}')
    joined = frame.merge(evaluation[['sample_id', 'abstract']], on='sample_id',
                         validate='many_to_one', suffixes=('', '_expected'))
    require((joined.abstract == joined.abstract_expected).all(), 'Wrong candidate source')


### `select_candidates()`

Keep the first five distinct summaries among ten returned outputs per paper.

#### Purpose and inputs

Only five are considered for the candidate evaluation, however, previous runs have procured duplicated candidates. To remedy this, 5 extra candidates have been generated for each paper to serve as backup candidates should duplicates occur.

1. `pool` is the raw generation table, with ten rows per paper
2. `evaluation` supplies the paper IDs and expected abstracts
3. `.itertuples(index=False)` visits its evaluation row
- selects that paper's raw outputs, converts `raw_rank` to integers, and sorts them numerically
4. `raw_rank` is the model's returned order **not ranking

#### How duplicates are detected

1. repeated/leading/trailing whitespace  are removed for comparison
2. A set named `seen` records normalized texts already accepted
3. A set named `selected` records distinct candidates
4. if text is already in `seen`, skip; else, append to `selected`

#### Five retained candidates

- The first five distinct outputs are retained
- `candidate_id` is reassigned 0-4 in acceptance order
- `raw_rank` preserves the original returned position

The loop stops once five are accepted. If all ten raw outputs contain fewer than five distinct normalized texts, the function raises an error.

Finally, `validate_candidates()` checks the complete table, including five distinct summaries for each paper and matching source abstracts. The returned DataFrame has 3,090 rows. It is returned to `generate_candidates()`.


In [3]:
def select_candidates(pool, evaluation):
    """Retain the first five distinct texts from each paper's ten raw outputs."""
    selected = []
    for row in evaluation.itertuples(index=False):
        group = pool[pool.sample_id == row.sample_id].copy()
        group['rank_number'] = group.raw_rank.astype(int)
        group = group.sort_values('rank_number')
        seen = set()
        for candidate in group.itertuples(index=False):
            normalized = ' '.join(candidate.raw_summary.split())
            if normalized in seen:
                continue
            seen.add(normalized)
            selected.append({'sample_id': row.sample_id, 'candidate_id': str(len(seen) - 1),
                             'raw_rank': str(candidate.rank_number), 'abstract': row.abstract,
                             'candidate_summary': candidate.raw_summary})
            if len(seen) == 5:
                break
        require(len(seen) == 5,
                f'Fewer than five distinct candidates among ten outputs for {row.sample_id}')
    frame = pd.DataFrame(selected)
    validate_candidates(frame, evaluation)
    return frame


### `generate_candidates()`

Generate ten beam outputs per paper, save progress, and retain five distinct candidates.

#### Purpose and inputs

This function generates alternatives for the 618 abstracts
1. Returns ten raw beam-search outputs per abstract
2. Calls `select_candidates()` to select five distinct summaries (see description for select_candidates())

The function loads evaluation pairs and checks the configured input limit is 512 tokens. 

Model
1. if `outputs/model/config.json` exists, then use local model directory; else, download `SamHuertas/SciTLDR-trained-BART` into `outputs/hub_models/SciTLDR-trained-BART/`
2. `BART_REVISION`, if provided, requests a Hub revision. Download metadata records the resolved revision.

Device and reproducibility
- Generation selects CUDA when available, otherwise CPU
- `set_seed(42)` and deterministic settings reduce certain sources of variation
- `.to(device)` moves the model or tensors to the chosen device
- `.eval()` sets evaluation behavior, such as disabling training-mode dropout
- `torch.inference_mode()` separately avoids gradient tracking

| Argument | Meaning |
|---|---|
| `num_beams=10` | Search with ten beam hypotheses |
| `num_return_sequences=10` | Return ten completed sequences per abstract |
| `do_sample=False` | Use beam search rather than random sampling |
| `max_new_tokens=64` | Limit newly generated token count |
| `min_new_tokens=0` | Impose no positive minimum here |
| `no_repeat_ngram_size=3` | Prevent repeated token trigrams within a generated sequence |
| `repetition_penalty=1.0` | Neutral multiplicative repetition penalty |
| `length_penalty=1.0` | Beam-search length-normalization parameter |
| `early_stopping=False` | Use the beam-search stopping heuristic |

#### Generation loop and checkpointing

- `evaluation.itertuples(index=False)` supplies each paper's ID and abstract
- Tokenization truncates the abstract to at most 512 model tokens and produces PyTorch tensors
- `model.generate(**batch, **decoding)` expands the input and decoding dictionaries into named arguments
- `batch_decode(..., skip_special_tokens=True)` converts output IDs back into readable text without control tokens.

Each paper must produce ten nonblank outputs. The code saves those ten rows to `generation_pool.csv` before proceeding. A completed raw paper group has ranks 0-9. On resumption, existing groups are checked for IDs, source text, and complete ranks; completed papers are skipped.

#### Final artifacts

When raw generation finishes, `select_candidates()` filters the pool. `generated_candidates.csv` saves five distinct summaries per paper, and `candidate_policy.json` describes the retention rule. Expected totals are 6,180 raw rows and 3,090 retained rows.


In [4]:
def generate_candidates():
    """Use trained BART to generate ten summaries and retain five distinct candidates per paper."""
    import os
    import json
    from pathlib import Path
    import pandas as pd
    import torch
    import yaml
    from huggingface_hub import snapshot_download
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

    evaluation = load_eval()
    cfg = yaml.safe_load((ROOT / 'configs' / 'config.yaml').read_text(encoding='utf-8'))
    require(cfg['data']['max_source_len'] == 512, 'Unexpected source-token limit')
    local = ROOT / 'outputs' / 'model'
    if (local / 'config.json').is_file():
        model_path = local
        identity = {'source': 'outputs/model'}
    else:
        revision = os.environ.get('BART_REVISION', '')
        hub_id = 'SamHuertas/SciTLDR-trained-BART'
        # A local directory uses ordinary files, avoiding Windows cache symlink privileges.
        model_path = Path(snapshot_download(
            hub_id, revision=revision or None,
            local_dir=ROOT / 'outputs' / 'hub_models' / 'SciTLDR-trained-BART'))
        from huggingface_hub._local_folder import read_download_metadata
        download = read_download_metadata(model_path, 'config.json')
        identity = {'source': hub_id,
                    'revision': download.commit_hash if download else revision or 'default'}
    weight_files = list(model_path.glob('*.safetensors')) + list(model_path.glob('pytorch_model*.bin'))
    require(bool(weight_files), 'Model weights are missing; do not substitute untrained BART')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    set_seed(42)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True).to(device).eval()
    decoding = dict(num_beams=10, num_return_sequences=10, do_sample=False,
        max_new_tokens=64, min_new_tokens=0, no_repeat_ngram_size=3,
        repetition_penalty=1.0, length_penalty=1.0, early_stopping=False)
    manifest(RUN / 'generation_manifest.json', {
        'input_sha256': sha256(RUN / 'evaluation_pairs.csv'), 'model': identity,
        'seed': 42, 'device': device, 'dtype': str(next(model.parameters()).dtype),
        'source_limit': 512, 'batch_size': 1,
        'decoding_overrides': decoding, 'model_generation_defaults': model.generation_config.to_dict(),
        'distinctness': 'case-sensitive whitespace-normalized exact text',
        'deduplication': 'first five distinct among ten raw returns; fail if fewer',
        'baseline_included_by_force': False,
    })
    output = RUN / 'generation_pool.csv'
    if output.exists():
        pool = read_csv(output)
        require(not pool.duplicated(['sample_id', 'raw_rank']).any(), 'Duplicate saved raw keys')
        for sample_id, group in pool.groupby('sample_id', sort=False):
            require(sample_id in set(evaluation.sample_id), 'Unknown saved sample')
            require(set(group.raw_rank.astype(int)) == set(range(10)), 'Incomplete raw group')
            expected = evaluation.set_index('sample_id').loc[sample_id, 'abstract']
            require((group.abstract == expected).all(), 'Saved abstract changed')
            require(group.raw_summary.map(lambda x: bool(x.strip())).all(), 'Blank saved output')
    else:
        pool = pd.DataFrame(columns=['sample_id', 'raw_rank', 'abstract', 'raw_summary'])
    done = set(pool.sample_id)
    for row in evaluation.itertuples(index=False):
        if row.sample_id in done:
            continue
        batch = tokenizer(row.abstract, truncation=True, max_length=512,
                          return_tensors='pt').to(device)
        with torch.inference_mode():
            sequences = model.generate(**batch, **decoding)
        texts = [text.strip() for text in tokenizer.batch_decode(sequences, skip_special_tokens=True)]
        require(len(texts) == 10 and all(texts), f'Generation failure: {row.sample_id}')
        part = pd.DataFrame([{'sample_id': row.sample_id, 'raw_rank': str(rank),
            'abstract': row.abstract, 'raw_summary': text} for rank, text in enumerate(texts)])
        pool = pd.concat([pool, part], ignore_index=True)
        atomic_csv(pool, output)  # One complete paper per checkpoint; no regeneration on resume.
        print('Generated papers:', pool.sample_id.nunique(), '/', len(evaluation), flush=True)

    frame = select_candidates(pool, evaluation)
    atomic_csv(frame, RUN / 'generated_candidates.csv')
    write_json({'returned_per_paper': 10, 'retained_per_paper': 5,
                'duplicates_allowed': False,
                'selection': 'first five distinct whitespace-normalized texts in raw rank order'},
               RUN / 'candidate_policy.json')
    print('Final candidates:', len(frame))


## Execution and results


## Generation model

Uses `outputs/model/` when available; otherwise downloads `SamHuertas/SciTLDR-trained-BART`. Set an optional Hub revision below.


In [5]:
BART_REVISION = ''  # Optional Hub revision; blank uses the current repository snapshot.


In [6]:
os.environ.update({'BART_REVISION': BART_REVISION,
                   'CUBLAS_WORKSPACE_CONFIG': ':4096:8'})
generate_candidates()
evaluation = load_eval()
candidate_path = RUN / 'generated_candidates.csv'
if candidate_path.exists():
    generated = read_csv(candidate_path)
    validate_candidates(generated, evaluation)
    print('Validated candidates:', len(generated))
    display(generated[['sample_id', 'candidate_id', 'candidate_summary']].head())


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Generated papers: 1 / 618
Generated papers: 2 / 618
Generated papers: 3 / 618
Generated papers: 4 / 618
Generated papers: 5 / 618
Generated papers: 6 / 618
Generated papers: 7 / 618
Generated papers: 8 / 618
Generated papers: 9 / 618
Generated papers: 10 / 618
Generated papers: 11 / 618
Generated papers: 12 / 618
Generated papers: 13 / 618
Generated papers: 14 / 618
Generated papers: 15 / 618
Generated papers: 16 / 618
Generated papers: 17 / 618
Generated papers: 18 / 618
Generated papers: 19 / 618
Generated papers: 20 / 618
Generated papers: 21 / 618
Generated papers: 22 / 618
Generated papers: 23 / 618
Generated papers: 24 / 618
Generated papers: 25 / 618
Generated papers: 26 / 618
Generated papers: 27 / 618
Generated papers: 28 / 618
Generated papers: 29 / 618
Generated papers: 30 / 618
Generated papers: 31 / 618
Generated papers: 32 / 618
Generated papers: 33 / 618
Generated papers: 34 / 618
Generated papers: 35 / 618
Generated papers: 36 / 618
Generated papers: 37 / 618
Generated 

,sample_id,candidate_id,candidate_summary
0,SJ1Xmf-Rb,0,"We propose FearNet, a generative model that do..."
1,SJ1Xmf-Rb,1,We propose FearNet for incremental class learn...
2,SJ1Xmf-Rb,2,"We propose FearNet, a generative model that do..."
3,SJ1Xmf-Rb,3,"We propose FearNet, a generative model that do..."
4,SJ1Xmf-Rb,4,"We propose FearNet, a generative model that do..."
